# 11 — Capstone: Mixture-of-Experts routing and load balancing

**Papers**
- Shazeer et al. (2017), *Outrageously Large Neural Networks: The Sparsely-Gated Mixture-of-Experts Layer*, §2.1 (Eq. 1, 3–5)
- Fedus, Zoph & Shazeer (2021), *Switch Transformers*, §2.2 (Eq. 4–6, the auxiliary load-balancing loss)
- Jiang et al. (2024), *Mixtral of Experts*, §2.1 (top-2 routing as used in a modern LLM)

This notebook uses everything from the earlier ones, with fewer hints. Try each exercise with the recipe from notebook 00 before looking at the solution.

**You will learn**
- turning "keep the top k, set the rest to $-\infty$" into tensor ops
- the **dense reference → sparse implementation** pattern: write the obviously correct but wasteful version first, then the efficient one, and test them against each other
- reading a loss where only part of the expression is differentiable, and why the paper made it that way

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from p2t import check, check_grad, seed

seed(0)

## 1. Top-k gating

Shazeer et al., Eq. 1 and 3 (the noise term of Eq. 4 is left out here):
$$y = \sum_{i=1}^{n} G(x)_i\,E_i(x) \qquad G(x) = \mathrm{Softmax}\big(\mathrm{KeepTopK}(H(x), k)\big)$$
$$\mathrm{KeepTopK}(v, k)_i = \begin{cases} v_i & \text{if } v_i \text{ is in the top } k \text{ elements of } v\\ -\infty & \text{otherwise}\end{cases}$$

$H(x) = x\,W_g$ are the router logits. After the softmax, non-selected experts have gate exactly 0, so they **don't need to be computed at all**. That's where MoE gets its efficiency.

### Exercise 1 — `topk_gating`
Input router logits `(N, E)`; output gates `(N, E)` with exactly `k` non-zeros per row that sum to 1. Hint: `torch.topk`, then `scatter` or `masked_fill`.

In [ ]:
def topk_gating(logits, k):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
logits = torch.randn(10, 6)
g = topk_gating(logits, 2)
check("rows sum to 1", g.sum(-1), torch.ones(10))
check("exactly k non-zeros per row", (g > 0).sum(-1), torch.full((10,), 2))
# Literal reference: for each row, sort, keep the top k, softmax over those
ref = torch.zeros(10, 6)
for n in range(10):
    idx = sorted(range(6), key=lambda e: -logits[n, e].item())[:2]
    vals = torch.softmax(logits[n, idx], 0)
    for j, e in enumerate(idx):
        ref[n, e] = vals[j]
check("topk_gating", g, ref)
check_grad("topk_gating", lambda l: topk_gating(l, 2), lambda l: torch.softmax(
    l.masked_fill(l < l.topk(2, -1).values[:, -1:], float("-inf")), -1), logits)

## 2. The MoE forward pass: dense reference vs. sparse implementation

**Dense reference** (the equation, literally): run *every* expert on *every* token and take the gate-weighted sum. It's correct and easy to verify, but it costs $E\times$ the compute.

**Sparse implementation:** for each expert, find the tokens routed to it, run the expert on only those tokens, scale by the gate, and **scatter-add** the results back into the output. `torch.Tensor.index_add_` does the scatter-add. (Production systems add capacity limits and all-to-all communication, but this is the core.)

### Exercise 2 — write both

In [ ]:
class Expert(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.w1, self.w2 = nn.Linear(d, h), nn.Linear(h, d)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)))


def moe_dense(x, gates, experts):
    """x: (N, D), gates: (N, E). Every expert on every token."""
    # YOUR CODE HERE
    raise NotImplementedError


def moe_sparse(x, gates, experts):
    """Same output as moe_dense, but each expert only processes the tokens routed to it."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
N, D, E, Hh, k = 64, 16, 8, 32, 2
experts = nn.ModuleList([Expert(D, Hh) for _ in range(E)])
Wg = torch.randn(D, E) * 0.5
x = torch.randn(N, D)
gates = topk_gating(x @ Wg, k)
check("moe_sparse == moe_dense", moe_sparse(x, gates, experts), moe_dense(x, gates, experts), atol=1e-5)
check_grad("moe_sparse grads == moe_dense grads (wrt x)",
           lambda x: moe_sparse(x, topk_gating(x @ Wg, k), experts),
           lambda x: moe_dense(x, topk_gating(x @ Wg, k), experts), x)

### Exercise 3 — count the FLOPs you saved
Count the tokens each expert processes in the sparse version. What fraction of the dense compute is used?

In [ ]:
def tokens_per_expert(gates):
    """-> (E,) long"""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
tpe = tokens_per_expert(gates)
check("each token goes to exactly k experts", tpe.sum(), torch.tensor(N * k))
print("tokens per expert:", tpe.tolist(), f"-> sparse uses {k}/{E} = {k / E:.0%} of the dense expert compute")

## 3. The load-balancing loss

A learned router tends to **collapse**: a few experts win early, receive more gradient, get better, and win more often. The Switch Transformer adds an auxiliary loss (Eq. 4–6):
$$\mathrm{loss} = \alpha\cdot N\cdot\sum_{i=1}^{N} f_i\cdot P_i$$
$$f_i = \frac{1}{T}\sum_{x\in\mathcal{B}}\mathbb{1}\{\mathrm{argmax}\ p(x) = i\} \qquad P_i = \frac{1}{T}\sum_{x\in\mathcal{B}} p_i(x)$$

**Decode it.** Switch Transformer notation differs from Shazeer's:
- here $N$ is the **number of experts** (it's `E` in our code) and $T$ is the **number of tokens** in the batch
- $p(x) = \mathrm{softmax}(\text{router logits})$, the **full** softmax before any top-k
- $f_i$ is the fraction of tokens *dispatched* to expert $i$ (top-1 in Switch). It comes from an argmax, so it's **not differentiable**.
- $P_i$ is the average router *probability* for expert $i$. It **is** differentiable.

The paper states that under uniform routing the loss equals $\alpha$ (its minimum): $f_i = P_i = 1/N$ gives $\alpha\cdot N\cdot N\cdot\frac{1}{N^2} = \alpha$.

### Exercise 4

In [ ]:
def switch_aux_loss(router_logits, alpha=0.01):
    """router_logits: (T, E) -> scalar"""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
T_, E_ = 12, 4
rl = torch.randn(T_, E_)
p = torch.softmax(rl, -1)
f_loop = torch.tensor([sum(1.0 for t in range(T_) if p[t].argmax() == i) / T_ for i in range(E_)])
P_loop = torch.tensor([sum(p[t, i].item() for t in range(T_)) / T_ for i in range(E_)])
check("switch_aux_loss", switch_aux_loss(rl, 1.0), E_ * (f_loop * P_loop).sum())

balanced = torch.eye(E_).repeat(3, 1) * 50   # each expert gets exactly 3 tokens, with near one-hot probs
check("perfectly balanced -> loss = alpha", switch_aux_loss(balanced, 0.01), torch.tensor(0.01), atol=1e-6)
collapsed = torch.zeros(T_, E_); collapsed[:, 0] = 50
check("fully collapsed -> loss = alpha * E", switch_aux_loss(collapsed, 0.01), torch.tensor(0.01 * E_), atol=1e-6)

### Experiment — the aux loss alone fixes a collapsed router

Start with a router biased toward expert 0, so it's collapsed, and minimize **only** the aux loss. Watch $f$ (the load) spread out. Consider: gradient flows only through $P$, so how does the load $f$ end up balanced? (Answer in the solutions.)

In [ ]:
seed(0)
x = torch.randn(512, D)
router = nn.Linear(D, E)
with torch.no_grad():
    router.bias.zero_(); router.bias[0] = 3.0   # collapse toward expert 0
opt = torch.optim.Adam(router.parameters(), lr=1e-2)
history = []
for step in range(300):
    logits = router(x)
    loss = switch_aux_loss(logits, alpha=1.0)
    opt.zero_grad(); loss.backward(); opt.step()
    history.append(F.one_hot(logits.argmax(-1), E).float().mean(0).detach())
history = torch.stack(history)
plt.stackplot(range(len(history)), history.T, labels=[f"expert {i}" for i in range(E)])
plt.xlabel("step"); plt.ylabel("fraction of tokens"); plt.legend(loc="upper right", fontsize=7); plt.show()
print("final load:", [f"{v:.2f}" for v in history[-1]])
assert history[-1].max() < 0.3, "load should be spread across experts"
print("✅ router un-collapsed")

## Reflection
1. Mixtral computes `softmax(topk(logits))` (renormalize over the chosen k), while Switch computes `topk(softmax(logits))` (keep the original probabilities, top-1). For $k = 1$, what gate value does each produce? Why might Switch *want* a gate that isn't 1?
2. Why does multiplying by $N$ in the aux loss matter? (Hint: what happens to the loss value as you scale up the number of experts?)
3. Real MoE layers have an **expert capacity** $C = \frac{\text{tokens}}{E}\cdot\text{capacity factor}$ and drop tokens beyond it. Where would you add that to `moe_sparse`?